In [1]:
%useLatestDescriptors
%use dataframe, kandy

// %use dataframe(1.0.0-Beta4n), kandy(0.8.3)

In [2]:
import util.Helpers

val helpers = Helpers()

In [3]:
fun queryByTimePeriodAndEntries(startYear: String, endYear: String, liftersPerWeightclass: Int) =
    """
SELECT
    pd.*
FROM
    powerlifting_data pd
        JOIN
    (
        SELECT
            meetname,
            date,
            weightclasskg,
            division,
            COUNT(*) AS lifter_count
        FROM
            powerlifting_data
        WHERE
            event = 'SBD' AND place != 'NS'
            AND date BETWEEN '$startYear-01-01' AND '$endYear-12-31'
            AND totalkg IS NOT NULL
        GROUP BY
            meetname, date, weightclasskg, division
        HAVING
            COUNT(*) >= $liftersPerWeightclass
    ) AS qualified_classes
    ON pd.meetname = qualified_classes.meetname
        AND pd.date = qualified_classes.date
        AND pd.weightclasskg = qualified_classes.weightclasskg
        AND pd.division = qualified_classes.division
WHERE
pd.event = 'SBD' AND pd.place != 'NS'
  AND pd.date BETWEEN '$startYear-01-01' AND '$endYear-12-31'
  AND pd.totalkg IS NOT NULL;
    """

In [4]:
import java.util.Date
import org.jetbrains.kotlinx.dataframe.annotations.*

// With @DataSchema, you get a generated typed API giving you compile-time type safety and IDE autocomplete.

@DataSchema
interface LifterData {
    val place: String
    val meetname: String
    val date: Date
    val event: String
    val weightclasskg: String?
    val entries: Int
    val division: String?
    val squat1kg: Double?
    val squat2kg: Double?
    val squat3kg: Double?
    val bench1kg: Double?
    val bench2kg: Double?
    val bench3kg: Double?
    val deadlift1kg: Double?
    val deadlift2kg: Double?
    val deadlift3kg: Double?
    val best3squatkg: Double?
    val best3benchkg: Double?
    val best3deadliftkg: Double?
    val totalkg: Double?
    val successfulLifts: Int
}

In [5]:
val query = queryByTimePeriodAndEntries(startYear = "2025", endYear = "2025", liftersPerWeightclass = 3)
val data: DataFrame<*> = helpers.fetchResults(query)

In [6]:
data.size()

47985 x 42

In [7]:
fun addNumberOfSuccessfulLifts(data: DataFrame<LifterData>, firstPlaceOnly: Boolean = true): DataFrame<LifterData> {

    val filteredDf = if (firstPlaceOnly) data.filter { it.place == "1" } else data

    return filteredDf.add("successfulLifts") {
        listOf(
            squat1kg, squat2kg, squat3kg,
            bench1kg, bench2kg, bench3kg,
            deadlift1kg, deadlift2kg, deadlift3kg
        ).count { it != null && it > 0.0 }
    }
        .filter { listOf(best3squatkg, best3benchkg, best3deadliftkg).all { it != null} }
        .groupBy { successfulLifts } // countPlot?
        .aggregate {
            count().into("count")
        }
        .filter { successfulLifts >= 3 }
        .sortBy { successfulLifts }
}

In [8]:
val winnersDataFrame = addNumberOfSuccessfulLifts(data.cast<LifterData>())

winnersDataFrame

successfulLifts,count
3,35
4,117
5,512
6,1284
7,2122
8,2546
9,1645


In [9]:
val barPlot = plot(winnersDataFrame) {

    bars {
        x(successfulLifts)
        y(count)
    }
}
barPlot
//barPlot.save("distribution-of-winners.svg")
//barPlot.save("distribution-of-winners.png")

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="F0HzEZ" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("F0HzEZ");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
"successfulLifts":[3.0,4.0,5.0,6.0,7.0,8.0,9.0],
"count":[35.0,117.0,512.0,1284.0,2122.0,2546.0,1645.0]
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"successfulLifts",
"y":"count"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"int",
"column":"successfulLifts"
},{
"type":"int",
"column":"count"
}]
},
"spec_id":"2"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 
 
 7 
 
 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 
 
 9 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 1,500 
 
 
 
 
 
 
 2,000 
 
 
 
 
 
 
 2,500 
 
 
 
 
 
 
 
 
 count 
 
 
 
 
 successfulLifts

In [10]:
val dfNoCount = data.filter { it.place == "1" }
    .add("successfulLifts") {
        listOf(
            squat1kg, squat2kg, squat3kg,
            bench1kg, bench2kg, bench3kg,
            deadlift1kg, deadlift2kg, deadlift3kg
        ).count { it != null && it > 0.0 }
    }
    .filter { listOf(best3squatkg, best3benchkg, best3deadliftkg).all { it != null} }


In [11]:
val countPlot = dfNoCount
    .filter { "successfulLifts"<Int>() in 3..9 }
    .plot {
        countPlot("successfulLifts")
    }

countPlot

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="MNl2Vj" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("MNl2Vj");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"successfulLifts",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"count"
},
"stat":"identity",
"data":{
"count":[1284.0,117.0,2122.0,2546.0,35.0,1645.0,512.0],
"x":[6.0,4.0,7.0,8.0,3.0,9.0,5.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"int",
"column":"x"
},{
"type":"int",
"column":"count"
}]
}
}],
"spec_id":"5"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 
 
 7 
 
 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 
 
 9 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 1,500 
 
 
 
 
 
 
 2,000 
 
 
 
 
 
 
 2,500 
 
 
 
 
 
 
 
 
 count 
 
 
 
 
 successfulLifts

In [12]:
val plotGridCount = plotGrid(
    listOf(barPlot, countPlot),
    nCol = 2
)

plotGridCount

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="7keEJl" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("7keEJl");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"layout":{
"name":"grid",
"ncol":2,
"nrow":1,
"fit":true,
"align":false
},
"figures":[{
"layers":[{
"mapping":{
"x":"successfulLifts",
"y":"count"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"mapping":{
},
"data":{
"successfulLifts":[3.0,4.0,5.0,6.0,7.0,8.0,9.0],
"count":[35.0,117.0,512.0,1284.0,2122.0,2546.0,1645.0]
},
"kind":"plot",
"data_meta":{
"series_annotations":[{
"type":"int",
"column":"successfulLifts"
},{
"type":"int",
"column":"count"
}]
},
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"spec_id":"9"
},{
"layers":[{
"mapping":{
"x":"x",
"y":"count"
},
"stat":"identity",
"data":{
"x":[6.0,4.0,7.0,8.0,3.0,9.0,5.0],
"count":[1284.0,117.0,2122.0,2546.0,35.0,1645.0,512.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"int",
"column":"x"
},{
"type":"int",
"column":"count"
}]
}
}],
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"successfulLifts",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"spec_id":"10"
}],
"kind":"subplots"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 
 
 7 
 
 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 
 
 9 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 1,500 
 
 
 
 
 
 
 2,000 
 
 
 
 
 
 
 2,500 
 
 
 
 
 
 
 
 
 count 
 
 
 
 
 successfulLifts 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 
 
 7 
 
 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 
 
 9 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 1,500 
 
 
 
 
 
 
 2,000 
 
 
 
 
 
 
 2,500 
 
 
 
 
 
 
 
 
 count 
 
 
 
 
 successfulLifts

In [13]:
kandyConfig.themeApplied = false

val plotWinners = winnersDataFrame.plot {

    bars {
        x(successfulLifts) {
            axis {
                name = "Successful Attempts"
                breaks(listOf(3, 4, 5, 6, 7, 8, 9), format = "d")
            }
        }
        y(count) {
            axis {
                name = "Number of Winners"
                breaks(listOf(500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000), format = "d")
            }
        }
        fillColor(count) {
            legend.type = LegendType.None
            scale = continuous(Color.hex("#dd4560")..Color.hex("#7e52ff"))
        }
        borderLine {
            color = Color.hex("#777777")
            width = 0.5
        }
    }
    layout {
        title = "Distribution of Winners by Successful Attempts"
        caption = "Data: Open IPF Meets 2025"
        size = 600 to 400
        style {
            global {
                text {
                    fontFamily = FontFamily.custom("Helvetica Neue")
                }
                plotCanvas {
                    title {
                        hJust = 0.5
                        margin = Margin(10.0)
                        fontSize = 16.0
                    }
                    caption {
                        hJust = 1.0
                        margin = Margin(10.0, 0.0, 0.0, 0.0)
                        fontSize = 12.0
                    }
                    margin = Margin(10.0, 30.0, 10.0, 10.0)
                }
            }
        }
    }
}

plotWinners
// Alternatively you can save your chart as an svg or png
//plotWinners.save("distribution-of-winners-custom-formatting.svg")
//plotWinners.save("distribution-of-winners-custom-formatting.png")

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="7fOAbr" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("7fOAbr");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Distribution of Winners by Successful Attempts"
},
"mapping":{
},
"data":{
"successfulLifts":[3.0,4.0,5.0,6.0,7.0,8.0,9.0],
"count":[35.0,117.0,512.0,1284.0,2122.0,2546.0,1645.0]
},
"ggsize":{
"width":600.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"breaks":[3.0,4.0,5.0,6.0,7.0,8.0,9.0],
"name":"Successful Attempts",
"format":"d",
"limits":[null,null]
},{
"aesthetic":"y",
"breaks":[500.0,1000.0,1500.0,2000.0,2500.0,3000.0,3500.0,4000.0,4500.0,5000.0],
"name":"Number of Winners",
"format":"d",
"limits":[null,null]
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_gradient",
"high":"#7e52ff",
"low":"#dd4560",
"limits":[null,null],
"guide":"none"
}],
"layers":[{
"mapping":{
"x":"successfulLifts",
"y":"count",
"fill":"count"
},
"stat":"identity",
"color":"#777777",
"size":0.5,
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"caption":{
"text":"Data: Open IPF Meets 2025"
},
"theme":{
"axis_ontop":false,
"plot_caption":{
"size":12.0,
"hjust":1.0,
"margin":[10.0,0.0,0.0,0.0],
"blank":false
},
"plot_margin":[10.0,30.0,10.0,10.0],
"text":{
"family":"Helvetica Neue",
"blank":false
},
"plot_title":{
"size":16.0,
"hjust":0.5,
"margin":[10.0,10.0,10.0,10.0],
"blank":false
},
"axis_ontop_y":false,
"axis_ontop_x":false
},
"data_meta":{
"series_annotations":[{
"type":"int",
"column":"successfulLifts"
},{
"type":"int",
"column":"count"
}]
},
"spec_id":"14"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 
 
 7 
 
 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 
 
 9 
 
 
 
 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 1000 
 
 
 
 
 
 
 1500 
 
 
 
 
 
 
 2000 
 
 
 
 
 
 
 2500 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Distribution of Winners by Successful Attempts 
 
 
 
 
 Number of Winners 
 
 
 
 
 Successful Attempts 
 
 
 
 
 Data: Open IPF Meets 2025

In [14]:
val allLiftersDataFrame = addNumberOfSuccessfulLifts(data.cast<LifterData>(), false)

In [15]:
allLiftersDataFrame

successfulLifts,count
3,277
4,1248
5,3798
6,7913
7,11571
8,12251
9,6873


In [16]:
val dfRatioWinners =
    dataFrameOf(
        winnersDataFrame.rename("count").into("winners").columns() +
                allLiftersDataFrame.select("count").rename("count").into("allLifters").columns()
    )
        .add("ratioWinners") {
            (column<Double>("winners").getValue(it) / column<Double>("allLifters").getValue(it)) * 100.0
        }

In [17]:
dfRatioWinners

successfulLifts,winners,allLifters,ratioWinners
3,35,277,"12,635379"
4,117,1248,"9,375000"
5,512,3798,"13,480779"
6,1284,7913,"16,226463"
7,2122,11571,"18,338951"
8,2546,12251,"20,781977"
9,1645,6873,"23,934235"


In [18]:
kandyConfig.themeApplied = false

val plotRatioWinners = dfRatioWinners.plot {

    bars {
        x(successfulLifts) {
            axis {
                name = "Successful Attempts"
                breaks(listOf(3, 4, 5, 6, 7, 8, 9), format = "d")
            }
        }
        y(ratioWinners) {
            axis.name = "Percentage"
            axis {
                breaks(listOf(5.0, 10.0, 15.0, 20.0, 25.0), format = "{.0f}%")
            }
        }
        fillColor(successfulLifts) {
            legend.type = LegendType.None
            scale = continuous(Color.hex("#dd4560")..Color.hex("#7e52ff"))
        }
        borderLine {
            color = Color.hex("#777777")
            width = 0.5
        }
    }
    layout {
        title = "Percentage of First Places by Successful Lifts"
        subtitle = "At least 3 lifters in Weight Class"
        caption = "Data: Open IPF meets 2025"
        size = 600 to 400
        style {
            global {
                text {
                    fontFamily = FontFamily.custom("Helvetica Neue")
                }
                plotCanvas {
                    title {
                        hJust = 0.5
                        margin = Margin(10.0)
                        fontSize = 14.0
                    }
                    subtitle {
                        hJust = 0.5
                        margin = Margin(5.0)
                        fontSize = 11.0
                    }
                    caption {
                        hJust = 1.0
                        margin = Margin(10.0, 0.0, 0.0, 0.0)
                    }
                    margin = Margin(10.0, 30.0, 10.0, 10.0)
                }
            }
        }
    }
}

plotRatioWinners
//plotRatioWinners.save("percetage-of-first-places.svg")

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="24OwH3" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("24OwH3");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Percentage of First Places by Successful Lifts",
"subtitle":"At least 3 lifters in Weight Class"
},
"mapping":{
},
"data":{
"successfulLifts":[3.0,4.0,5.0,6.0,7.0,8.0,9.0],
"ratioWinners":[12.63537906137184,9.375,13.480779357556608,16.226462782762543,18.33895082533921,20.7819769814709,23.9342354139386]
},
"ggsize":{
"width":600.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"breaks":[3.0,4.0,5.0,6.0,7.0,8.0,9.0],
"name":"Successful Attempts",
"format":"d",
"limits":[null,null]
},{
"aesthetic":"y",
"breaks":[5.0,10.0,15.0,20.0,25.0],
"name":"Percentage",
"format":"{.0f}%",
"limits":[null,null]
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_gradient",
"high":"#7e52ff",
"low":"#dd4560",
"limits":[null,null],
"guide":"none"
}],
"layers":[{
"mapping":{
"x":"successfulLifts",
"y":"ratioWinners",
"fill":"successfulLifts"
},
"stat":"identity",
"color":"#777777",
"size":0.5,
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"caption":{
"text":"Data: Open IPF meets 2025"
},
"theme":{
"axis_ontop":false,
"plot_caption":{
"hjust":1.0,
"margin":[10.0,0.0,0.0,0.0],
"blank":false
},
"plot_margin":[10.0,30.0,10.0,10.0],
"text":{
"family":"Helvetica Neue",
"blank":false
},
"plot_title":{
"size":14.0,
"hjust":0.5,
"margin":[10.0,10.0,10.0,10.0],
"blank":false
},
"axis_ontop_y":false,
"axis_ontop_x":false,
"plot_subtitle":{
"size":11.0,
"hjust":0.5,
"margin":[5.0,5.0,5.0,5.0],
"blank":false
}
},
"data_meta":{
"series_annotations":[{
"type":"int",
"column":"successfulLifts"
},{
"type":"float",
"column":"ratioWinners"
}]
},
"spec_id":"17"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 
 
 7 
 
 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 
 
 9 
 
 
 
 
 
 
 
 
 
 
 5% 
 
 
 
 
 
 
 10% 
 
 
 
 
 
 
 15% 
 
 
 
 
 
 
 20% 
 
 
 
 
 
 
 25% 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Percentage of First Places by Successful Lifts 
 
 
 
 
 At least 3 lifters in Weight Class 
 
 
 
 
 Percentage 
 
 
 
 
 Successful Attempts 
 
 
 
 
 Data: Open IPF meets 2025

In [21]:
val plotGrid = plotGrid(
    listOf(plotWinners, plotRatioWinners),
    nCol = 2,
    widths = listOf(500.0, 500.0),
    heights =  listOf(400.0, 400.0)
)

plotGrid
//plotGrid.save("plot-grid-example.svg")

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="koDcYl" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("koDcYl");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"layout":{
"name":"grid",
"ncol":2,
"nrow":1,
"widths":[500.0,500.0],
"heights":[400.0,400.0],
"fit":true,
"align":false
},
"figures":[{
"ggtitle":{
"text":"Distribution of Winners by Successful Attempts"
},
"mapping":{
},
"data":{
"successfulLifts":[3.0,4.0,5.0,6.0,7.0,8.0,9.0],
"count":[35.0,117.0,512.0,1284.0,2122.0,2546.0,1645.0]
},
"ggsize":{
"width":600.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"breaks":[3.0,4.0,5.0,6.0,7.0,8.0,9.0],
"name":"Successful Attempts",
"format":"d",
"limits":[null,null]
},{
"aesthetic":"y",
"breaks":[500.0,1000.0,1500.0,2000.0,2500.0,3000.0,3500.0,4000.0,4500.0,5000.0],
"name":"Number of Winners",
"format":"d",
"limits":[null,null]
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_gradient",
"high":"#7e52ff",
"low":"#dd4560",
"limits":[null,null],
"guide":"none"
}],
"layers":[{
"mapping":{
"x":"successfulLifts",
"y":"count",
"fill":"count"
},
"stat":"identity",
"color":"#777777",
"size":0.5,
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"caption":{
"text":"Data: Open IPF Meets 2025"
},
"theme":{
"axis_ontop":false,
"plot_caption":{
"size":12.0,
"hjust":1.0,
"margin":[10.0,0.0,0.0,0.0],
"blank":false
},
"plot_margin":[10.0,30.0,10.0,10.0],
"text":{
"family":"Helvetica Neue",
"blank":false
},
"plot_title":{
"size":16.0,
"hjust":0.5,
"margin":[10.0,10.0,10.0,10.0],
"blank":false
},
"axis_ontop_y":false,
"axis_ontop_x":false
},
"data_meta":{
"series_annotations":[{
"type":"int",
"column":"successfulLifts"
},{
"type":"int",
"column":"count"
}]
},
"spec_id":"29"
},{
"ggtitle":{
"text":"Percentage of First Places by Successful Lifts",
"subtitle":"At least 3 lifters in Weight Class"
},
"mapping":{
},
"data":{
"successfulLifts":[3.0,4.0,5.0,6.0,7.0,8.0,9.0],
"ratioWinners":[12.63537906137184,9.375,13.480779357556608,16.226462782762543,18.33895082533921,20.7819769814709,23.9342354139386]
},
"ggsize":{
"width":600.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"breaks":[3.0,4.0,5.0,6.0,7.0,8.0,9.0],
"name":"Successful Attempts",
"format":"d",
"limits":[null,null]
},{
"aesthetic":"y",
"breaks":[5.0,10.0,15.0,20.0,25.0],
"name":"Percentage",
"format":"{.0f}%",
"limits":[null,null]
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_gradient",
"high":"#7e52ff",
"low":"#dd4560",
"limits":[null,null],
"guide":"none"
}],
"layers":[{
"mapping":{
"x":"successfulLifts",
"y":"ratioWinners",
"fill":"successfulLifts"
},
"stat":"identity",
"color":"#777777",
"size":0.5,
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"caption":{
"text":"Data: Open IPF meets 2025"
},
"theme":{
"axis_ontop":false,
"plot_caption":{
"hjust":1.0,
"margin":[10.0,0.0,0.0,0.0],
"blank":false
},
"plot_margin":[10.0,30.0,10.0,10.0],
"text":{
"family":"Helvetica Neue",
"blank":false
},
"plot_title":{
"size":14.0,
"hjust":0.5,
"margin":[10.0,10.0,10.0,10.0],
"blank":false
},
"axis_ontop_y":false,
"axis_ontop_x":false,
"plot_subtitle":{
"size":11.0,
"hjust":0.5,
"margin":[5.0,5.0,5.0,5.0],
"blank":false
}
},
"dat

## Bonus analysis! Which was the most commonly missed lift?

In [91]:
val attemptColumns = listOf(
    LifterData::squat1kg, LifterData::squat2kg, LifterData::squat3kg,
    LifterData::bench1kg, LifterData::bench2kg, LifterData::bench3kg,
    LifterData::deadlift1kg, LifterData::deadlift2kg, LifterData::deadlift3kg
)

In [92]:
fun addWhichLiftsWereMissed(df: DataFrame<LifterData>): DataFrame<*> {

    return df.add("successfulLifts") {
        listOf(
            squat1kg, squat2kg, squat3kg,
            bench1kg, bench2kg, bench3kg,
            deadlift1kg, deadlift2kg, deadlift3kg
        ).count { it != null && it > 0.0 }
    }
        .filter { successfulLifts == 8 }
        .let { frame ->
            attemptColumns.map { lift ->
                val missedCount = frame[lift.name].cast<Double?>().count { it != null && it <= 0.0 }
                dataFrameOf("lift", "totalMissed")(lift.name, missedCount)
            }
                .concat()
        }
}

In [89]:
val missedLiftsDataFrame = addWhichLiftsWereMissed(data.cast<LifterData>())


In [90]:
missedLiftsDataFrame

lift,totalMissed
squat1kg,498
squat2kg,559
squat3kg,2042
bench1kg,345
bench2kg,677
bench3kg,4064
deadlift1kg,223
deadlift2kg,370
deadlift3kg,3346


In [77]:
val liftLabels = mapOf(
    "squat1kg" to "S1", "squat2kg" to "S2", "squat3kg" to "S3",
    "bench1kg" to "B1", "bench2kg" to "B2", "bench3kg" to "B3",
    "deadlift1kg" to "D1", "deadlift2kg" to "D2", "deadlift3kg" to "D3"
)

val plotMissedLifts = missedLiftsDataFrame
    .convert { lift }.with { liftLabels[it] ?: it }
    .plot {
    bars {
        x(lift)
        y(totalMissed) { axis.name = "count of attempts missed" }
        fillColor(totalMissed) {
            legend.type = LegendType.None
            scale = continuous(Color.hex("#dd4560")..Color.hex("#7e52ff"))
        }
        borderLine {
            color = Color.hex("#777777")
            width = 0.5
        }
    }
    layout {
        title = "Most Commonly Missed Lift - All lifters"
        caption = "Data: Open IPF meets 2025"
        size = 600 to 400
        xAxisLabel = "Lift"
        style {
            global {
                text {
                    fontFamily = FontFamily.custom("Helvetica Neue")
                }
                plotCanvas {
                    title {
                        hJust = 0.5
                        margin = Margin(10.0)
                        fontSize = 17.0
                    }
                    subtitle {
                        hJust = 0.5
                        margin = Margin(5.0)
                    }
                    caption {
                        hJust = 1.0
                        margin = Margin(10.0, 0.0, 0.0, 0.0)
                    }
                    margin = Margin(5.0, 30.0, 20.0, 5.0)
                }
            }
        }
    }
}
plotMissedLifts
//plotMissedLifts.save("most-commonly-missed-lifts-2015-2024.svg")


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="fhpeoC"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("fhpeoC");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Most Commonly Missed Lift - All lifters"
},
"mapping":{
},
"guides":{
"x":{
"title":"Lift"
}
},
"data":{
"lift":["S1","S2","S3","B1","B2","B3","D1","D2","D3"],
"totalMissed":[498.0,559.0,2069.0,345.0,677.0,4122.0,223.0,370.0,3388.0]
},
"ggsize":{
"width":600.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"name":"count of attempts missed",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"lift",
"y":"totalMissed"
},
"stat":"identity",
"color":"#777777",
"size":0.5,
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"fill":"#fec92e",
"data":{
}
}],
"caption":{
"text":"Data: Open IPF meets 2025"
},
"theme":{
"axis_ontop":false,
"plot_caption":{
"hjust":1.0,
"margin":[10.0,0.0,0.0,0.0],
"blank":false
},
"plot_margin":[5.0,30.0,20.0,5.0],
"text":{
"family":"Helvetica Neue",
"blank":false
},
"plot_title":{
"size":17.0,
"hjust":0.5,
"margin":[10.0,10.0,10.0,10.0],
"blank":false
},
"axis_ontop_y":false,
"axis_ontop_x":false,
"plot_subtitle":{
"hjust":0.5,
"margin":[5.0,5.0,5.0,5.0],
"blank":false
}
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"lift"
},{
"type":"int",
"column":"totalMissed"
}]
},
"spec_id":"53"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 S1 
 
 
 
 
 
 
 
 
 S2 
 
 
 
 
 
 
 
 
 S3 
 
 
 
 
 
 
 
 
 B1 
 
 
 
 
 
 
 
 
 B2 
 
 
 
 
 
 
 
 
 B3 
 
 
 
 
 
 
 
 
 D1 
 
 
 
 
 
 
 
 
 D2 
 
 
 
 
 
 
 
 
 D3 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 2,000 
 
 
 
 
 
 
 3,000 
 
 
 
 
 
 
 4,000 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Most Commonly Missed Lift - All lifters 
 
 
 
 
 count of attempts missed 
 
 
 
 
 Lift 
 
 
 
 
 Data: Open IPF meets 2025

Answer: 3rd attempt bench press, it's not even close!